## RAG(retrival Argumented Generation)

In [1]:
pip install --upgrade langchain

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install langchain_community

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install Unstructured[pdf]

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install langchain_groq

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install -qU langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [9]:
os.environ['GROQ_API_KEY'] = "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"

In [10]:
import requests
url = 'https://www.tutorialspoint.com/object_oriented_python/object_oriented_python_tutorial.pdf'
response= requests.get(url)

In [11]:
with open('python_oops.pdf', 'wb') as file:
    file.write(response.content)

In [12]:
# Load the Document
loader = PyPDFLoader('python_oops.pdf')
documents = loader.load()
print(f"Loaded {len(documents)} pages")

Loaded 111 pages


In [13]:
# Split documents into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 111 chunks


In [14]:
# Create embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")
print("Vector store created successfully")

Vector store created successfully


In [17]:
# Set up the LLM and RAG chain
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=1)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
{context}

Question: {question}""")

qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG chain ready")

RAG chain ready


In [19]:
# Query the RAG pipeline
query = "How many types of inheritanceP?"
result = qa_chain.invoke(query)
print("Answer:", result)

Answer: The provided context does not mention "types of inheritance". However, according to OOP fundamentals mentioned in the provided context (which discusses inheritance but does not specifically mention types of inheritance), there are typically three main types of inheritance: 

1. **Single Inheritance**: When a child class inherits from only one parent class.
2. **Multi-level Inheritance**: When a child class inherits from a parent class that is a subclass of another class.
3. **Multiple Inheritance**: When a child class inherits properties and behavior from more than one parent class.

However, the context itself does not explicitly mention these types of inheritance.


In [20]:
pip install gradio

  Using cached pydantic-2.11.10-py3-none-any.whl.metadata (68 kB)
Using cached pydantic-2.11.10-py3-none-any.whl (444 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------- ----------------------- 0.8/2.0 MB 4.9 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 6.9 MB/s eta 0:00:00

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.41.5

    Uninstalling pydantic_core-2.41.5:

      Successfully uninstalled pydantic_core-2.41.5

   ---------------------------------------- 0/2 [pydantic-core]
   ---------------------------------------- 0/2 [pydantic-core]
   ---------------------------------------- 0/2 [pydantic-core]
   ---------------------------------------- 0/2 [pydantic-core]
   ---------------------------------------- 0/2 [pydantic-core]
   ---------------------------------------- 0/2 [pydantic-core]
   ---------------------------------------- 0/2 [pydantic-core]
   ------------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 2.0.10 requires langchain-core<0.4.0,>=0.3.37, but you have langchain-core 1.2.22 which is incompatible.


In [22]:
import gradio as gr

def answer_question(question):
    if not question.strip():
        return "Please enter a question."
    result = qa_chain.invoke(question)
    return result

demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(lines=3, placeholder="Ask a question about the PDF...", label="Your Question"),
    outputs=gr.Textbox(lines=10, label="Answer"),
    title="RAG - PDF Q&A",
    description="Ask questions about the Python OOPs PDF using Retrieval-Augmented Generation (Groq LLaMA + ChromaDB)",
    examples=[
        ["What are the types of inheritance in Python?"],
        ["What is encapsulation?"],
        ["Explain polymorphism with an example."],
    ]
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
